SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

# Lab 1: Vision Language Models

## Cosmos 3 Reasoner

In this notebook, we'll explore Vision Language Models (VLMs) hands-on using **Cosmos 3**, NVIDIA's reasoning VLM for physical AI. We'll bring up the model as a NIM microservice from its pre-seeded cache, run inference on a traffic-camera frame and a short clip, learn how to prompt it effectively, measure its performance, and free the GPU again - everything you need before moving on to dataset generation and fine-tuning in the next notebooks.

**Estimated time: 30 minutes.** Parts 1-8 are the core path; Part 9 (benchmarking) is optional if you are behind schedule. Part 10 (tear down) is **required** before opening Lab 2.

<div class="alert alert-block alert-warning">
<b>Do not use "Run All".</b> Every inference cell calls a live GPU service, Part 9 issues many requests, and Part 10 stops the model server. Run the cells one at a time, top to bottom, and read each output before moving on. Every cell is safe to re-run.
</div>

### Learning Objectives:
This notebook explores the following topics:
* What VLMs are and the Cosmos 3 model family
* How the Cosmos 3 Reasoner NIM is deployed, and how to make sure it is running from a pre-seeded cache and healthy
* Single image and video inference through the OpenAI library
* How vision token count affects output quality and latency
* Good vs bad prompts for VLMs
* Structured outputs: formatted reports, JSON, true/false and multiple choice
* Reasoning vs non-reasoning mode
* Benchmarking (optional): latency, throughput, and the knobs that drive them
* How to tear down the deployment and verify the GPU is free for the next notebook


### Table of Contents

**[Set Up the Environment](#Set-Up-the-Environment)**  
**[Part 1: Understanding VLMs and Cosmos 3](#Part-1:-Understanding-VLMs-and-Cosmos-3)**  
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 What is a VLM?](#1.1-What-is-a-VLM?)  
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 The Cosmos 3 Family](#1.2-The-Cosmos-3-Family)  
**[Part 2: Deploy Cosmos 3 Reasoner](#Part-2:-Deploy-Cosmos-3-Reasoner)**  
**[Part 3: Image Inference](#Part-3:-Image-Inference)**  
**[Part 4: Video Inference](#Part-4:-Video-Inference)**  
**[Part 5: Vision Tokens - Quality vs Latency](#Part-5:-Vision-Tokens---Quality-vs-Latency)**  
**[Part 6: Good vs Bad Prompts](#Part-6:-Good-vs-Bad-Prompts)**  
**[Part 7: Structured Outputs and Tasks](#Part-7:-Structured-Outputs-and-Tasks)**  
&nbsp;&nbsp;&nbsp;&nbsp;[7.1 Formatted Reports](#7.1-Formatted-Reports)  
&nbsp;&nbsp;&nbsp;&nbsp;[7.2 JSON Events](#7.2-JSON-Events)  
&nbsp;&nbsp;&nbsp;&nbsp;[7.3 True-False and Multiple Choice](#7.3-True-False-and-Multiple-Choice)  
**[Part 8: Reasoning vs Non-Reasoning](#Part-8:-Reasoning-vs-Non-Reasoning)**  
**[Part 9: Benchmarking Performance (optional)](#Part-9:-Benchmarking-Performance-(optional))**  
&nbsp;&nbsp;&nbsp;&nbsp;[9.1 Latency: Time to First Token and Output Length](#9.1-Latency:-Time-to-First-Token-and-Output-Length)  
&nbsp;&nbsp;&nbsp;&nbsp;[9.2 Input Size: Sampled Frames](#9.2-Input-Size:-Sampled-Frames)  
&nbsp;&nbsp;&nbsp;&nbsp;[9.3 Concurrency and Throughput](#9.3-Concurrency-and-Throughput)  
**[Part 10: Tear Down](#Part-10:-Tear-Down)**  
**[Review](#Review)**  

### Time Budget (30-minute slot; about 26 minutes without the optional Part 9)

| Section | Minutes | What runs |
|---|---|---|
| Set Up the Environment | 2 | env block, imports, media check (nothing is installed or downloaded) |
| Part 1: VLMs and Cosmos 3 | 3 | reading |
| Part 2: Deploy / wait for ready | 1 (up to 5 if the NIM is still loading) | ensure-running cell + readiness poll |
| Part 3: Image Inference | 3 | 2 model calls |
| Part 4: Video Inference | 3 | 1 model call |
| Part 5: Vision Tokens | 3 | 2 model calls |
| Part 6: Good vs Bad Prompts | 3 | 2 model calls |
| Part 7: Structured Outputs | 4 | 4 model calls |
| Part 8: Reasoning vs Non-Reasoning | 3 | 2 model calls (the reasoning call can take up to ~1 min) |
| Part 9: Benchmarking - **optional** | 5 | ~14 model calls; skip if fewer than 7 minutes remain |
| Part 10: Tear Down + Review | 1 | **must run** before Lab 2 |


---
### Set Up the Environment

*Estimated time: 2 minutes*


Cosmos 3 is deployed as a **NIM microservice** and exposes an **OpenAI-compatible API**. This means we can interact with it using the standard `openai` Python library, which is already installed in this lab image - nothing is installed or downloaded at runtime, and no API key of any kind is needed.

In this environment the NIM runs as a container inside the lab's inner Docker daemon (Docker-in-Docker) and is reached by that service's hostname and published port. Every path, endpoint, GPU index and container name used in this notebook is read once, below, from an environment variable with a sensible default, so the same notebook runs unchanged on the DLI platform and on your own machine.

Let's start by defining our endpoint, container settings and test media.


In [ ]:
import os

# ---- ENV / CONFIG: the single place for every path, endpoint, GPU index and container name ----
# Values are injected by the course compose; the defaults below are the DLI production layout.

# The Cosmos 3 Reasoner NIM runs inside the lab's inner Docker daemon ("dind"). From this notebook it is
# reached by that service's hostname and its published port - never a loopback/bind address of this container.
# The course compose exports COURSE_VLM_BASE_URL (prefixed on purpose so it never leaks into a VSS deploy);
# VLM_BASE_URL is an explicit override for running elsewhere.
VLM_BASE_URL = os.environ.get("VLM_BASE_URL") or os.environ.get("COURSE_VLM_BASE_URL", "http://dind:30082")   # Cosmos 3 Reasoner NIM (OpenAI-compatible)
DOCKER_HOST = os.environ.get("DOCKER_HOST", "tcp://dind:2375")             # inner Docker daemon (Docker-in-Docker)
DOCKER_API = DOCKER_HOST.replace("tcp://", "http://")                      # same daemon, spoken to over its HTTP API

# NIM container settings (used by the ensure-running cell in Part 2 and the tear-down cell in Part 10)
NIM_CONTAINER = os.environ.get("NIM_CONTAINER", "cosmos3-reasoner")
NIM_IMAGE = os.environ.get("NIM_IMAGE", "nvcr.io/nim/nvidia/cosmos3-reasoner:1.7")   # pre-pulled by the hotlab
NIM_CACHE = os.environ.get("NIM_CACHE", "/dli/task/data/cosmos3-reasoner-cache")   # pre-seeded by the course loader (S3); a docker volume name also works
NIM_GPU = os.environ.get("NIM_GPU", "1")                                              # GPU index the NIM is pinned to
NIM_PORT = os.environ.get("NIM_PORT", "30082")                                        # published port on the dind host
NIM_MODEL_SIZE = os.environ.get("NIM_MODEL_SIZE", "nano")                             # nano | super
NIM_READY_TIMEOUT_S = int(os.environ.get("NIM_READY_TIMEOUT_S", "600"))               # max wait for /v1/health/ready

# Course-owned media staged by the DLI loader at learner launch (see MEDIA.md for provenance).
LAB_DATA_DIR = os.environ.get("LAB_DATA_DIR", "/dli/task/data")
test_image = os.path.join(LAB_DATA_DIR, "traffic.png")        # one traffic-camera frame
test_video = os.path.join(LAB_DATA_DIR, "traffic_short.mp4")  # 8-30 s traffic-camera clip

print(f"VLM endpoint : {VLM_BASE_URL}")
print(f"Docker daemon: {DOCKER_HOST}")
print(f"NIM container: {NIM_CONTAINER}  (image {NIM_IMAGE}, cache {NIM_CACHE}, GPU {NIM_GPU})")
print(f"Media        : {test_image}, {test_video}")


In [ ]:
# All Python dependencies are pre-installed in the lab image (requirements.txt is consumed at image build time).
# Nothing is installed at runtime. This cell only imports them and prints the versions so a broken image fails loudly.
import json
import time
import html
import base64
import shutil
import subprocess
import importlib.metadata as _md

import requests
import openai
from openai import OpenAI
from IPython.display import Markdown, display, Image, Video

for _pkg in ("openai", "requests", "ipython"):
    print(f"{_pkg} {_md.version(_pkg)}")


In [ ]:
# Sanity-check the pre-staged course media (nothing is downloaded at runtime).
for _p in (test_image, test_video):
    if not os.path.isfile(_p):
        raise FileNotFoundError(
            f"Missing course asset: {_p}\n"
            f"LAB_DATA_DIR is {LAB_DATA_DIR}. The course loader should have staged this file at launch - "
            "ask an instructor/TA to check the loader logs."
        )
    print(f"OK  {_p}  ({os.path.getsize(_p) / 1e6:.1f} MB)")


The following helper functions will verify API responses, talk to the inner Docker daemon, and base64-encode media for the API. `check_response` will help debug if any errors occur.


In [ ]:
# Helper: verify HTTP responses
def check_response(response, text=False):
    print(f"Response Code: {response.status_code}")
    if response.status_code == 200:
        print("Response Status: Success")
        if text:
            print(response.text)
            return response.text
        else:
            print(json.dumps(response.json(), indent=4))
            return response.json()
    else:
        print("Response Status: Error")
        print(response.text)
        return None


# Helper: minimal Docker Engine API client against the inner (dind) daemon at DOCKER_HOST.
# This is exactly what the `docker` CLI does under the hood; using the HTTP API directly means the lab
# image needs neither the docker CLI nor the docker Python SDK.
# Verified on the devbox and in production (Sep 3): the lab container reaches the Engine API at DOCKER_API.
def docker_api(method, path, timeout=60, **kw):
    return requests.request(method, f"{DOCKER_API}{path}", timeout=timeout, **kw)


def container_state(name):
    """'running' / 'exited' / 'created' / ... or None if the container does not exist."""
    r = docker_api("GET", f"/containers/{name}/json")
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()["State"]["Status"]


# Helper: base64-encode any binary file (image or video) for the API
def encode_file(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


encode_image = encode_file  # kept for readability where the file is an image


---
### Part 1: Understanding VLMs and Cosmos 3

*Estimated time: 3 minutes (reading)*


#### 1.1 What is a VLM?

A **Vision Language Model (VLM)** is an AI model that accepts both visual inputs (images, video frames) and text. You provide it with an image or video clip along with a text prompt, and it responds with text. Internally, a vision encoder converts pixels into tokens that the language model processes alongside your prompt. The important distinction: VLMs can interpret visual content, which a standard text-only Large Language Model (LLM) cannot do.

**Cosmos 3** is NVIDIA's open omnimodal world model family for physical AI and robotics. The capabilities most relevant to this lab are:

- **Spatial-temporal understanding**: it reasons about space, time, and physics across video frames, not just individual images
- **Chain-of-thought reasoning**: using `<think>` tags, the model works through complex scenes step by step before producing a final answer
- **Dense captioning** with **temporal localization**: it produces detailed descriptions of video scenes and can output timestamped events in JSON format
- Cosmos 3 also supports 2D grounding (bounding boxes) and embodied reasoning (next-action prediction), though those capabilities are outside the scope of this lab

#### 1.2 The Cosmos 3 Family

The full Cosmos 3 model actually contains **two towers** in every transformer layer: an autoregressive **Reasoner** tower (the VLM) and a diffusion **Generator** tower (for video generation). The Reasoner tower can run standalone (the Generator always activates both towers), and what we deploy in this lab is the **Reasoner tower only** - this is what the "Reasoner" NIM serves. You will meet the Generator side of Cosmos in the next notebook, where we use it to generate and augment video datasets.

Cosmos 3 comes in three sizes:

| Model | Total Params | Reasoner Tower | Base Architecture | Target Platform |
|-------|-------------|----------------|-------------------|-----------------|
| Cosmos3-Edge | 4B | 2B | From scratch (Nemotron-style) | Jetson / on-device |
| Cosmos3-Nano | 16B | 8B | Qwen3-VL-8B | Workstation |
| Cosmos3-Super | 64B | 32B | Qwen3-VL-32B | Datacenter |

This is why you will see the same model referred to as both "Nano (16B)" and "Nano Reasoner (8B)" - the NIM checkpoint contains just the 8B Reasoner tower of the 16B model. In this lab, we use **Nano**.

---
### Part 2: Deploy Cosmos 3 Reasoner

*Estimated time: 1 minute if the NIM is already up; up to 5 minutes if it still has to load weights onto the GPU*

The Cosmos 3 Reasoner NIM serves both the Nano and Super variants from a single container image (`nvcr.io/nim/nvidia/cosmos3-reasoner`); the size is selected at startup with the `NIM_MODEL_SIZE` environment variable (`nano` or `super`).

Deploying a NIM takes one `docker run` command. For reference, this is the deployment this lab uses - pinned to a single GPU, with the model cache in a pre-seeded directory:

```bash
docker run -d --name cosmos3-reasoner -u 0 \
    --gpus '"device=1"' --shm-size=32GB \
    -e NIM_MODEL_SIZE=nano \
    -p 30082:8000 \
    -v /dli/task/data/cosmos3-reasoner-cache:/opt/nim/.cache \
    nvcr.io/nim/nvidia/cosmos3-reasoner:1.7
```

On your own machine you would add `-e NGC_API_KEY=...` the **first** time only, so the NIM can pull the model weights into the cache. **In this course the container image was pre-pulled and the model cache was pre-seeded by the course loader into `/dli/task/data/cosmos3-reasoner-cache`, so nothing is downloaded and no API key of any kind is needed or present in this environment.** Once the NIM is running, its API requires no authentication, which is why it is only reachable on the lab's internal Docker network.

The NIM normally starts when your lab launches. The next cell makes sure it is running - it is idempotent: if the container is already up it is reused; if it was stopped (for example by a previous run of Part 10, or by Lab 2) it is recreated from the cache. It also refuses to continue if the GPU it needs is still held by something else, so a half-torn-down previous notebook surfaces here instead of as an out-of-memory error later.


In [ ]:
# Ensure the Cosmos 3 Reasoner NIM is running (idempotent - safe to re-run at any time).
# Equivalent to the `docker run` above, issued to the inner Docker daemon over its HTTP API.

state = container_state(NIM_CONTAINER)
if state == "running":
    print(f"{NIM_CONTAINER} is already running - reusing it.")
else:
    # --- Preflight 1: the image and the pre-seeded cache volume must already exist (never pull or download here)
    if docker_api("GET", f"/images/{NIM_IMAGE}/json").status_code == 404:
        # The seed writes its progress to a file on a volume shared with this container; read it so a learner sees
        # "still pulling" or "failed at step X" instead of guessing.
        _seed_status_path = "/opt/vss/_course-runtime/seed-status"
        _seed_status = open(_seed_status_path).read().strip() if os.path.isfile(_seed_status_path) else "no status file yet (the seed has not started, or this is not the DLI platform)"
        if "FAILED" in _seed_status:
            raise RuntimeError(f"Image {NIM_IMAGE} is not in the inner daemon and the launch-time seed FAILED - ask a TA now, waiting will not help.\n"
                               f"Seed status: {_seed_status}")
        raise RuntimeError(f"Image {NIM_IMAGE} is not in the inner daemon yet. The course seeds it at launch; that takes several minutes on a "
                           f"fresh VM - wait and re-run this cell.\nSeed status: {_seed_status}")
    # The cache is a directory on the course data volume (visible here AND in dind at the same path), or a docker volume name.
    if NIM_CACHE.startswith("/"):
        # The loader copies the cache file by file, so a non-empty directory is not a complete cache.
        # The seeded cache is ~9.9 GB; anything well below that is still landing.
        _cache_bytes = sum(p.stat().st_size for p in __import__("pathlib").Path(NIM_CACHE).rglob("*") if p.is_file()) if os.path.isdir(NIM_CACHE) else 0
        if _cache_bytes < 9.5e9:
            raise RuntimeError(f"NIM cache {NIM_CACHE} holds {_cache_bytes / 1e9:.1f} GB of ~9.9 GB - the course loader is still staging it. "
                               "Wait a few minutes and re-run this cell.")
        if not (os.path.isdir(NIM_CACHE) and os.listdir(NIM_CACHE)):
            raise RuntimeError(f"NIM cache {NIM_CACHE} is missing or empty. The course loader stages it at launch - "
                               "wait for the loader to finish (a few minutes after the lab opens) or ask a TA.")
    elif docker_api("GET", f"/volumes/{NIM_CACHE}").status_code == 404:
        raise RuntimeError(f"Cache volume {NIM_CACHE} is missing. It should be pre-seeded at launch - ask a TA.")

    # --- Preflight 2: GPU NIM_GPU must be (mostly) free. vLLM reserves ~90% of the GPU, so if another
    #     process (e.g. a still-running Lab 2 kernel) holds it, the NIM would fail to start.
    if shutil.which("nvidia-smi"):
        out = subprocess.run(["nvidia-smi", f"--id={NIM_GPU}", "--query-gpu=memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True)
        if out.returncode == 0 and out.stdout.strip():
            used_mib, total_mib = (int(x) for x in out.stdout.strip().split(","))
            print(f"GPU {NIM_GPU}: {used_mib} / {total_mib} MiB in use before start")
            if used_mib > 0.25 * total_mib:
                raise RuntimeError(
                    f"GPU {NIM_GPU} still has {used_mib} MiB in use. Shut down other notebook kernels "
                    "(JupyterLab -> Running tab -> Shut Down All) and re-run this cell.")
    else:
        print("nvidia-smi not available in this container - skipping the GPU free-memory preflight.")

    # --- Remove a stopped/created leftover so the name is free
    if state is not None:
        print(f"Removing leftover container {NIM_CONTAINER} (state: {state})")
        docker_api("DELETE", f"/containers/{NIM_CONTAINER}", params={"force": "true"}).raise_for_status()

    # --- Create + start (mirrors the reference docker run; no real credential is passed)
    spec = {
        "Image": NIM_IMAGE,
        # Run as root: the cache is a bind-mounted directory created by the course loader (root). The NIM's default
        # user (uid 1000) cannot write there, which fails with "Permission denied" while downloading the manifest.
        "User": "0",
        "Env": [
            f"NIM_MODEL_SIZE={NIM_MODEL_SIZE}",   # honoured by NIM 1.7: the log shows "NIM_MODEL_SIZE=nano" selecting the Nano profiles
            "NGC_API_KEY=not-needed-cache-preseeded",  # placeholder only: with a complete cache the NIM never contacts NGC (verified on the devbox and in production)
        ],
        "ExposedPorts": {"8000/tcp": {}},
        "HostConfig": {
            "ShmSize": 32 * 1024**3,                                   # --shm-size=32GB
            "Binds": [f"{NIM_CACHE}:/opt/nim/.cache"],                 # -v /dli/task/data/cosmos3-reasoner-cache:/opt/nim/.cache
            "PortBindings": {"8000/tcp": [{"HostPort": str(NIM_PORT)}]},  # -p 30082:8000
            "DeviceRequests": [{"Driver": "", "Count": 0, "DeviceIDs": [str(NIM_GPU)], "Capabilities": [["gpu"]]}],  # --gpus '"device=1"'
        },
    }
    r = docker_api("POST", "/containers/create", params={"name": NIM_CONTAINER}, json=spec)
    if r.status_code != 201:
        raise RuntimeError(f"Could not create {NIM_CONTAINER}: HTTP {r.status_code} {r.text}")
    r = docker_api("POST", f"/containers/{NIM_CONTAINER}/start")
    if r.status_code not in (204, 304):
        raise RuntimeError(f"Could not start {NIM_CONTAINER}: HTTP {r.status_code} {r.text}")
    print(f"Started {NIM_CONTAINER} ({NIM_IMAGE}, size={NIM_MODEL_SIZE}) on GPU {NIM_GPU} from cache '{NIM_CACHE}'.")


Now let's wait until the model service reports ready. No download happens here: the NIM loads the cached weights onto the GPU and builds its inference engine, which takes a few minutes from a warm cache. The cell below polls the readiness endpoint every 5 seconds and returns as soon as it answers **HTTP 200**.

<div class="alert alert-block alert-info">
        <b>Expected:</b> "READY after Ns" followed by "Response Code: 200". If the cell is still waiting after ~5 minutes, or times out, raise your hand - do not restart anything yourself.
    </div>


In [ ]:
# Wait for the Cosmos 3 Reasoner NIM to become ready (returns immediately if it already is)
READY_URL = VLM_BASE_URL + "/v1/health/ready"
POLL_S = 5

t0 = time.time()
ready = False
while time.time() - t0 < NIM_READY_TIMEOUT_S:
    try:
        resp = requests.get(READY_URL, timeout=5)
        if resp.status_code == 200:
            ready = True
            break
        status = f"HTTP {resp.status_code}"
    except requests.RequestException as e:
        status = type(e).__name__
    cstate = container_state(NIM_CONTAINER)
    if cstate in ("exited", "dead", None):
        raise RuntimeError(f"NIM container {NIM_CONTAINER} is {cstate} - re-run the ensure-running cell above or ask a TA to check its logs.")
    print(f"  waiting... {int(time.time() - t0):4d}s  ({status}; container state: {cstate})")
    time.sleep(POLL_S)

if not ready:
    raise RuntimeError(f"NIM not ready after {NIM_READY_TIMEOUT_S}s - ask a TA to check the container logs.")

print(f"READY after {int(time.time() - t0)}s")
check_response(resp)


Now let's create an OpenAI client for the model and retrieve the configured model name.

In [ ]:
# Create OpenAI-compatible client (the NIM's API needs no key, but the client requires a non-empty string)
vlm_client = OpenAI(base_url=VLM_BASE_URL + "/v1", api_key="EMPTY")

# Retrieve available models
try:
    vlm_model = vlm_client.models.list().data[0].id
    print(f"VLM Model: {vlm_model}")
except Exception:
    raise RuntimeError("Could not reach the VLM endpoint - make sure the readiness cell above printed READY before continuing.") from None


---
### Part 3: Image Inference

*Estimated time: 3 minutes*

The simplest way to use Cosmos 3 is to pass a single image and ask a question about it. The image is sent as a base64-encoded data URL (or a public http(s) URL the NIM container can download) in the message content.

The frame below is a course-owned asset: a single frame from the synthetic traffic-camera dataset that was generated with NVIDIA Cosmos for this course (see `MEDIA.md` for provenance). All media in this lab is synthetic - there are no real people or licence plates in it.

Let's start with a basic image description.


In [ ]:
# Display the test image
Image(filename=test_image, width=800)

To send an image to the VLM, we use the OpenAI chat completions API with `image_url` content type. The API expects the image either as a base64-encoded data URL (what we do here, using the `encode_file` helper from the setup) or as an `http(s)://` URL that the NIM container itself can download. Local file paths are not accepted - the model server runs in its own container and cannot see this notebook's files.

<div class="alert alert-block alert-info">
        <b>Important:</b> In the message content, media (images/videos) should be listed <b>before</b> text to match the model's training format.
    </div>

For the captioning cells we use `temperature=0.4, top_p=0.4` - deliberately low-variance settings so that your outputs are easy to compare with your neighbour's. The single-token verdict cells in Part 7 use `0.2/0.2`, and reasoning mode in Part 8 uses the Cosmos 3 prompt guide's recommended `0.6/0.95`; each is noted where it appears.

One thing you may notice right away: Cosmos 3 sometimes reasons on its own. If an answer opens with a `<think>` block, that is the chain-of-thought you will study in Part 8; the final answer is the text after the closing `</think>` tag.

Every request in this notebook keeps the system prompt at the default `You are a helpful assistant.` and puts role, focus and format instructions in the user message. That is what the [Cosmos 3 Reasoner prompt guide](https://github.com/NVIDIA/cosmos/blob/main/cookbooks/cosmos3/reasoner/reasoner_prompt_guide.md) and the [NIM API examples](https://docs.nvidia.com/nim/vision-language-models/1.7.0/examples/cosmos-reason3/api.html) do, and the model is tuned for it.

In [ ]:
image_b64 = encode_file(test_image)

start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": "Describe this traffic scene in detail: the road layout, the vehicles (type, position, direction of travel), traffic signals and signage, and the weather and lighting conditions."},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)
response_time = time.time() - start_t

image_caption = response.choices[0].message.content
print(f"Response Time: {response_time:.2f} seconds\n")
print(image_caption)


We can also ask specific questions about the image. The VLM will ground its answers in what it actually sees.

In [ ]:
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": "You are a traffic operations monitor. This frame is from a fixed traffic camera. Is there an incident in this scene (a vehicle stopped in a live lane, a collision, debris, or pedestrians in the roadway)? If yes, which vehicle is involved and what is the evidence? If you cannot tell, say so."},
            ],
        },
    ],
    max_tokens=512,
    temperature=0.4,
    top_p=0.4,
)

print(response.choices[0].message.content)


---
### Part 4: Video Inference

*Estimated time: 3 minutes*

Cosmos 3 can also process video. When a video is provided, the model samples frames from it and processes them together, giving the model temporal understanding of what's happening over time.

- **Watch** the clip once before running inference, so you can judge whether the model's description is grounded in what actually happens. Like the frame, it is a course-owned Cosmos-generated traffic-camera clip.

<div class="alert alert-block alert-info">
        <b>Note:</b> By default, the NIM samples video at 4 FPS. Frame sampling can be adjusted per request through the <code>media_io_kwargs</code> parameter - we will use this in the benchmarking section.
    </div>


In [ ]:
# Preview the clip we are about to send to the model (embedded so it renders behind the lab proxy).
# Outputs are cleared before this notebook is committed, so the embedded bytes never end up in the repo.
_size_mb = os.path.getsize(test_video) / 1e6
if _size_mb <= 20:
    display(Video(test_video, width=800, embed=True, mimetype="video/mp4"))
else:
    print(f"{test_video} is {_size_mb:.0f} MB - too large to embed; open it from the file browser instead.")


For video inference, the `video_url` content type is used. Like the image, the clip is base64-encoded into a `data:video/mp4;base64,...` URL (or given as an `http(s)://` URL reachable from the NIM container). Keep clips short - the whole file travels inside the JSON request body.


In [ ]:
# The video is sent exactly like the image: read the bytes, base64-encode, wrap in a data URL.
video_data_url = f"data:video/mp4;base64,{encode_file(test_video)}"   # adjust the MIME type if the clip is not MP4
print(f"Encoded {os.path.basename(test_video)}: {len(video_data_url) / 1e6:.1f} MB in the request body")

# --- Send request ---
start_t = time.time()

response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": video_data_url
                    },
                },
                {
                    "type": "text",
                    "text": "Caption the video in detail. Describe the scene, the vehicles, and any events.",
                },
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)

response_time = time.time() - start_t

video_caption = response.choices[0].message.content

print(f"Response Time: {response_time:.2f} seconds\n")
print(video_caption)


---
### Part 5: Vision Tokens - Quality vs Latency

*Estimated time: 3 minutes*

When Cosmos 3 processes an image or video, it converts the visual input into **vision tokens**. The number of vision tokens directly impacts:

1. **Quality** - More tokens = higher resolution = finer details
2. **Latency** - More tokens = more processing time
3. **Memory** - More tokens = more GPU memory required

The vision token count is controlled by the image/video resolution and the number of frames:

```
Vision Tokens ≈ (Height × Width) / (32 × 32) × Frames / 2
```

The `/ 2` comes from temporal compression: Cosmos 3 merges pairs of adjacent frames into one set of tokens. For a single image, the frames and temporal terms drop out and the count is simply `(Height × Width) / (32 × 32)`.

For reference, here are some common configurations:

| Height | Width | Frames | Vision Tokens | Recommended Use |
|--------|-------|--------|---------------|----------------|
| 448    | 448   | 80     | ~7,840        | Production default shape |
| 540    | 960   | 30     | ~7,594        | Balanced |
| 720    | 1280  | 20     | ~9,000        | High quality |
| 1080   | 1920  | 8      | ~8,100        | Ultra-high res, fewer frames |

<div class="alert alert-block alert-info">
        <b>Recommendation:</b> Keep vision tokens below 16,000. Going beyond will significantly increase processing time and memory usage.
    </div>

Let's demonstrate this by running the same image at two pixel budgets and comparing the results. One rule shapes the experiment: **a budget above the frame's native resolution buys nothing** - the model just upsamples the same pixels and pays for more tokens. So the LOW budget deliberately discards most of the pixels (expect it to miss small things: signs, distant vehicles, lane markings), and the HIGH budget keeps the frame at native resolution. The budget is a request-level option (`extra_body={"mm_processor_kwargs": ...}`), and the API reports the resulting token count in `usage.prompt_tokens`, so we can prove that the knob did something.

**After running the two cells, here is what to look for:**
- Does the HIGH output name things (signs, lane markings, distant vehicles) that the LOW output missed?
- How many more prompt tokens did HIGH use, and how much longer did it take? (Several times more tokens - the count also includes the text prompt, so the ratio is below the 8x pixel ratio. Both runs are capped at the same short output length, so the time difference is the extra input processing; on a 480p frame that is a modest gap, and Part 9 shows it at scale with video.)
- Did either output invent something that is not in the frame?


In [ ]:
# LOW vision tokens - fast but fewer details
# The pixel budget is a top-level request option (extra_body -> mm_processor_kwargs), not part of the image item.
# One vision token covers a 32x32 patch = 1,024 pixels, so a budget of N tokens = N * 1024 pixels.
# Verified with NIM 1.7 (devbox and production): the "size" keys are accepted and usage.prompt_tokens changes with the budget.
PIX_PER_TOKEN = 32 * 32
# Budgets are set RELATIVE to the source frame: a VLM cannot see detail the pixels do not contain, so a budget above
# the image's native size only upsamples the same information (more tokens, no new detail). LOW throws away ~7/8 of
# the pixels so the loss of detail is visible; HIGH keeps the frame at its native resolution.
from PIL import Image as _PILImage
with _PILImage.open(test_image) as _im:
    _w, _h = _im.size
NATIVE_TOKENS = max(1, (_w * _h) // PIX_PER_TOKEN)
LOW_TOKENS  = max(64, NATIVE_TOKENS // 8)
HIGH_TOKENS = NATIVE_TOKENS
LOW_BUDGET  = {"size": {"shortest_edge": (LOW_TOKENS // 2) * PIX_PER_TOKEN, "longest_edge": LOW_TOKENS * PIX_PER_TOKEN}}
HIGH_BUDGET = {"size": {"shortest_edge": (HIGH_TOKENS // 2) * PIX_PER_TOKEN, "longest_edge": HIGH_TOKENS * PIX_PER_TOKEN}}
print(f"Frame {_w}x{_h} = ~{NATIVE_TOKENS} vision tokens at native resolution -> LOW budget ~{LOW_TOKENS} tokens, HIGH budget ~{HIGH_TOKENS} tokens")

# The prompt is byte-identical in the LOW and HIGH cells so the comparison isolates the vision-token budget.
detail_prompt = "List every vehicle, traffic signal, road marking and sign you can see in this traffic-camera frame, each with its approximate position. Answer directly as a compact bulleted list - no introduction, no reasoning."
DETAIL_MAX_TOKENS = 200   # the same short cap for both runs, so the two outputs are comparable in length and the timing reflects the input, not how much the model chose to write

start_t = time.time()
response_low = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": detail_prompt},
            ],
        },
    ],
    max_tokens=DETAIL_MAX_TOKENS,
    temperature=0.4,
    top_p=0.4,
    extra_body={"mm_processor_kwargs": LOW_BUDGET},
)
low_time = time.time() - start_t
low_output = response_low.choices[0].message.content
low_prompt_tokens = response_low.usage.prompt_tokens if response_low.usage else None

print(f"LOW Token Budget - Response Time: {low_time:.2f}s | prompt tokens (mostly vision): {low_prompt_tokens}")
print(f"Output:\n{low_output}")


In [ ]:
# HIGH vision tokens - slower but more details (same prompt, same image, only the budget changes)
start_t = time.time()
response_high = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": detail_prompt},
            ],
        },
    ],
    max_tokens=DETAIL_MAX_TOKENS,
    temperature=0.4,
    top_p=0.4,
    extra_body={"mm_processor_kwargs": HIGH_BUDGET},
)
high_time = time.time() - start_t
high_output = response_high.choices[0].message.content
high_prompt_tokens = response_high.usage.prompt_tokens if response_high.usage else None

print(f"HIGH Token Budget - Response Time: {high_time:.2f}s | prompt tokens (mostly vision): {high_prompt_tokens}")
print(f"Output:\n{high_output}")


In [ ]:
# Side by side. Model output is HTML-escaped so any markup the model emits is shown, not rendered.
markdown_string = f"""
<div style="display: flex; gap: 20px;">
  <div style="flex: 1;">
  <h2> Low Vision Tokens </h2>
  <h3> Prompt tokens: {low_prompt_tokens} &nbsp;|&nbsp; Response Time: {low_time:.2f}s </h3>
  <pre style="white-space: pre-wrap;">{html.escape(low_output or "")}</pre>
  </div>
  <div style="flex: 1;">
  <h2> High Vision Tokens </h2>
  <h3> Prompt tokens: {high_prompt_tokens} &nbsp;|&nbsp; Response Time: {high_time:.2f}s </h3>
  <pre style="white-space: pre-wrap;">{html.escape(high_output or "")}</pre>
  </div>
</div>
"""

Markdown(markdown_string)


Compare the two outputs. The LOW run should have lost or mis-described small elements (signs, lane markings, distant vehicles) that the HIGH run keeps; if both read the same, the frame is so simple that even 1/8 of the pixels suffices - try a busier frame. Going *above* native resolution would not have helped: the extra tokens would describe upsampled pixels, not new information. The prompt-token counts are the proof: the same image, the same prompt, but several times more tokens went into the model in the HIGH run.

This is the fundamental trade-off of every production VLM pipeline: more pixels and more frames produce better results, but at the cost of increased latency and memory. We will measure this directly in the benchmarking section.


---
### Part 6: Good vs Bad Prompts

*Estimated time: 3 minutes*

VLMs are only as good as the prompts you give them. A vague prompt produces a vague answer; a specific prompt that tells the model **what role to take**, **what to focus on**, and **what format to answer in** produces dramatically better results.

Let's run the same video through a bad prompt and a good prompt and compare. The prompts live in a small **presets** menu in the next cell:

- **Run** the presets cell as-is the first time.
- **Edit** `BAD_PRESET` / `GOOD_PRESET` (or add your own entry) and re-run the two inference cells to compare any pair.

**After running, here is what to look for:**
- Did the vague prompt produce anything a traffic operator could act on?
- Did the good prompt follow the numbered structure and say "not visible" where appropriate?
- Did either answer hallucinate an event that is not in the clip you watched in Part 4?


In [ ]:
# --- Prompt presets: pick one by name or add your own, then re-run the two cells below ---
# All presets target the traffic-anomaly use case that the VSS lab in Part 2 of the course (VSS alert verification) automates.
# bad_vague and good_inspection were run on the shipped clip (devbox and production, Sep 3); the two focus presets are untested there.
PROMPT_PRESETS = {
    "bad_vague": "What's in this video?",

    "good_inspection": """You are a traffic operations monitor reviewing an intersection camera clip.

Watch the video and report:
1. Every vehicle visible, its type (car, truck, bus, motorcycle), and what it is doing
2. Whether any pedestrians or cyclists are present in or near the roadway and whether the crossing is clear (count and location only - do not describe individuals)
3. Any traffic incidents or hazards (stalled vehicles, collisions, near-misses, red-light running, debris or blocked lanes), with the approximate time in the clip at which each occurs

Watch the whole clip to the end before answering - events can happen in the last seconds. Be specific about where in the frame each observation occurs. If something is not visible, say so - do not guess.""",

    "stalled_vehicle_focus": """You are a traffic operations monitor. Focus only on stopped vehicles.
For each vehicle that is not moving, state its lane position and whether it is stopped normally (at a signal, in a queue) or abnormally (in a live lane, on the median, after a crash).
If no vehicle is stopped abnormally, answer exactly: "No stalled vehicle.\"""",

    "collision_focus": """You are a traffic operations monitor. Focus only on collisions and near-misses.
Describe any contact or sudden evasive manoeuvre between vehicles, with the approximate time in the clip and the lanes involved.
If there is no collision or near-miss, answer exactly: "No collision.\"""",

    # Free-form slot: write your own prompt here. Expect roughly 1 in 3 free-form prompts to give a clean,
    # operator-usable answer on the first try - iterate on role, focus and format like the presets do.
    "custom": """You are a traffic operations monitor. <write your own instructions here>""",
}

BAD_PRESET = "bad_vague"           # <-- change these two names to compare any pair of presets
GOOD_PRESET = "good_inspection"

bad_prompt = PROMPT_PRESETS[BAD_PRESET]
good_prompt = PROMPT_PRESETS[GOOD_PRESET]
print(f"[{BAD_PRESET}]\n{bad_prompt}\n\n[{GOOD_PRESET}]\n{good_prompt}")


In [ ]:
# BAD prompt - vague, no role, no focus, no format
start_t = time.time()
response_bad = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": bad_prompt},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.2,   # low variance: the point of this cell is the prompt, not the sampling
    top_p=0.2,
)
bad_time = time.time() - start_t
bad_output = response_bad.choices[0].message.content

print(f"Response Time: {bad_time:.2f}s\n")
print(bad_output)


In [ ]:
# GOOD prompt - role, focus areas, and output format (from the presets cell above)
start_t = time.time()
response_good = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": good_prompt},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.2,   # low variance: the point of this cell is the prompt, not the sampling
    top_p=0.2,
)
good_time = time.time() - start_t
good_output = response_good.choices[0].message.content

print(f"Response Time: {good_time:.2f}s\n")
print(good_output)


Compare the two outputs. The bad prompt usually gets you a one-line description; the good prompt gets you an inspection report an operator can act on. Now check the report against the clip you watched: did it catch the event, at the right time? A well-structured answer can still be wrong - structure makes the output *usable*, it does not make it *true*. That gap is exactly why the VSS lab in Part 2 pairs the VLM with a verification step instead of trusting a single answer.

The pattern that works for VLM prompting:
- **Role**: tell the model who it is ("You are a traffic operations monitor")
- **Focus**: list exactly what to look for - the model reports what you ask about and skips what you don't
- **Format**: specify how the answer should be structured
- **Honesty**: instruct the model to say when something is not visible, which reduces hallucination

Keep this in mind for the next notebook as well - the same principles apply when writing prompts for video generation.

---
### Part 7: Structured Outputs and Tasks

*Estimated time: 4 minutes*

Free-text answers are fine for humans, but production systems need output they can parse. Cosmos 3 follows formatting instructions well, which lets us turn it into a component that emits **reports**, **JSON**, or **single-token verdicts** on demand.


#### 7.1 Formatted Reports

Let's ask for a structured markdown report instead of a free-form caption. Note how the prompt specifies the exact sections the report should contain.

In [ ]:
report_prompt = """You are a video analytics report generator. Watch the video and produce a traffic report in Markdown with exactly these sections:

## Summary
One paragraph describing the intersection and overall traffic activity.

## Timeline
A bulleted list of notable events in chronological order.

## Incidents & Hazards
A bulleted list of any traffic incidents or safety concerns. If there are none, state that explicitly."""

start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": report_prompt},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)
report_time = time.time() - start_t

print(f"Response Time: {report_time:.2f}s\n")
display(Markdown(response.choices[0].message.content))

---
#### 7.2 JSON Events

Cosmos 3 can produce **temporally localized** captions - describing events with timestamps in structured JSON format. This is the type of output that video analytics pipelines store and index for search.

In [ ]:
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": video_data_url
                    },
                },
                {"type": "text", "text": "Describe the notable events in the provided video. Provide the result in json format with 'mm:ss.ff' format for time depiction for each event. Use keywords 'start', 'end' and 'caption' in the json output."},
            ],
        },
    ],
    max_tokens=2048,
    temperature=0.4,
    top_p=0.4,
)

raw = response.choices[0].message.content
print(raw)

# Production check: is it parseable JSON? (strip a ```json fence if the model added one)
cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
try:
    events = json.loads(cleaned)
    events = events if isinstance(events, list) else events.get("events", [events])
    print(f"\nParsed {len(events)} event(s):")
    for ev in events:
        print(f"  {ev.get('start')} -> {ev.get('end')}: {ev.get('caption')}")
except (json.JSONDecodeError, AttributeError) as e:
    print(f"\nNot valid JSON ({e}) - in production you would retry or tighten the prompt.")


---
#### 7.3 True-False and Multiple Choice

The cheapest possible VLM output is a **single token**. True/false questions power real-time alerting (is the condition met, yes or no?), and multiple choice is the standard format for VLM evaluation benchmarks - you will meet it again in the fine-tuning notebook when we evaluate models.

Because the output is one token, these calls are also dramatically faster than captioning - compare the response times.

In Part 2 of the course, VSS alert verification asks exactly this kind of yes/no question about every chunk of a live camera stream.

**What to look for:** the verdict should be a single word, its response time should be a fraction of the captioning calls above, and it should agree with what you saw in the clip. Counting is a known weak spot for VLMs, so if the multiple-choice letter disagrees with what you saw, you have just found the kind of error that evaluation benchmarks are built to measure.


In [ ]:
# True/False - the alerting pattern
tf_question = "You are a traffic monitoring system. Does the video show any collision, near-miss, or stalled vehicle? Answer with only 'yes' or 'no'."

start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": tf_question},
            ],
        },
    ],
    max_tokens=10,
    temperature=0.2,
    top_p=0.2,
)
tf_time = time.time() - start_t

print(f"Response Time: {tf_time:.2f}s\n")
print(f"Question: {tf_question}")
print(f"Verdict: {response.choices[0].message.content}")

In [ ]:
# Multiple choice - the evaluation pattern
mcq_question = """How many vehicles are visible in the video?

A) None
B) 1-3
C) 4-10
D) More than 10

Answer with only the letter of the correct option."""

start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": mcq_question},
            ],
        },
    ],
    max_tokens=5,
    temperature=0.2,
    top_p=0.2,
)
mcq_time = time.time() - start_t

print(f"Response Time: {mcq_time:.2f}s\n")
print(f"Question: {mcq_question}\n")
print(f"Answer: {response.choices[0].message.content}")

---
### Part 8: Reasoning vs Non-Reasoning

*Estimated time: 3 minutes*

One of Cosmos 3's most powerful features is **chain-of-thought reasoning**. When enabled, the model first thinks through the problem step-by-step inside `<think>` tags before writing its final answer immediately after the `</think>` tag.

This is especially useful for:
- Complex scene understanding
- Safety analysis (incident detection, hazard identification)
- Causal reasoning (what happened and why)

Reasoning is prompt-driven: to enable it, we append the reasoning prompt template to the user message. Let's ask the same question **with and without** reasoning and compare both the answers and the cost.

<div class="alert alert-block alert-info"><b>Expect a few seconds to about a minute</b> for the reasoning call, depending on how long a trace the model writes. Let it finish; do not interrupt and re-run.</div>

**What to look for:** does the reasoning trace cite concrete evidence from the frame (which vehicle, which lane) before the final answer, and how many times longer did it take than the direct answer? Check the cited evidence against the frame itself - a trace reads convincingly even when it invents a detail, and that is exactly the failure a verification step has to catch.


In [ ]:
# The Cosmos 3 reasoning prompt template
REASONING_PROMPT = """Answer the question using the following format:

<think>
Your reasoning.
</think>

Write your final answer immediately after the </think> tag."""

print(REASONING_PROMPT)

In [ ]:
question = ("You are a traffic operator watching this camera. Is there an incident in this scene "
            "(a vehicle stopped in a live lane, a collision, debris, or pedestrians in the roadway)? "
            "Answer yes or no, then say which vehicle is involved and why, citing the evidence in the frame.")

# WITHOUT reasoning
start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": question},
            ],
        },
    ],
    max_tokens=512,
    temperature=0.4,
    top_p=0.4,
)
plain_time = time.time() - start_t
plain_output = response.choices[0].message.content
plain_tokens = response.usage.completion_tokens if response.usage else None

print(f"Response Time: {plain_time:.2f} seconds | completion tokens: {plain_tokens}\n")
print(plain_output)


In [ ]:
# WITH reasoning
start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": f"{question}\n\n{REASONING_PROMPT}"},
            ],
        },
    ],
    max_tokens=4096,
    temperature=0.6,
    top_p=0.95,
)
reasoning_time = time.time() - start_t
reasoning_output = response.choices[0].message.content
reasoning_tokens = response.usage.completion_tokens if response.usage else None

print(f"Response Time: {reasoning_time:.2f} seconds | completion tokens: {reasoning_tokens}\n")
print(reasoning_output)
print(f"\nCost of reasoning: {reasoning_time / max(plain_time, 1e-6):.1f}x the time, "
      f"{reasoning_tokens} vs {plain_tokens} generated tokens.")


Compare the two responses. Without reasoning, the model answers directly. With reasoning, it first works through its analysis inside the `<think>` tags before producing a concise final answer after the `</think>` tag - often more grounded on complex scenes, though not guaranteed to be - the trace can state evidence that is not in the frame, so verify it before trusting the conclusion, but at the cost of generating many more tokens (compare the response times and the completion-token counts).

That trade-off is the rule of thumb: **use reasoning when accuracy on complex judgments matters, skip it when speed matters** (like the single-token verdicts from Part 7).

<div class="alert alert-block alert-info">
<b>Tip:</b> When enabling reasoning, set <code>max_tokens</code> to 4096 (as in the cell above) so the trace is not cut off before the answer. The template and the sampling parameters used here (<code>temperature=0.6</code>, <code>top_p=0.95</code>) are the ones recommended in the Cosmos 3 Reasoner prompt guide, which recommends <code>temperature=0.7</code>, <code>top_p=0.8</code> for non-reasoning use. The NIM API reference shows an alternative template with an explicit <code>&lt;answer&gt;</code> tag - either works, but be consistent within one application.
</div>


---
### Part 9: Benchmarking Performance (optional)

*Estimated time: 5 minutes - **optional***

<div class="alert alert-block alert-info">
<b>Behind schedule?</b> Skip to <a href="#Part-10:-Tear-Down">Part 10</a> - the tear down is required, this section is not. The "What to expect" notes below each experiment let you follow the discussion without running it.
</div>

Before deploying a VLM in production, you need to know what it costs. Four knobs drive VLM performance:

- **ISL (input sequence length)**: mostly vision tokens - resolution × frames
- **OSL (output sequence length)**: how many tokens you ask the model to generate
- **Sampled frames**: how much of the video the model actually sees
- **Concurrency**: how many requests run at once

Let's measure each one.


In [ ]:
# Results of the live measurements below are recorded here, all with the same layout.
bench = {"gpu": "H100 NVL", "model": vlm_model, "clip": os.path.basename(test_video),
         "osl": {}, "fps": {}, "concurrency": {}}
print("Benchmark record initialised - run the cells below to measure.")

#### 9.1 Latency: Time to First Token and Output Length

Two latency numbers matter: **TTFT** (time to first token - dominated by input processing) and **total time** (dominated by output length). We can measure both with a streaming request.

In [ ]:
bench = globals().get("bench") or {"osl": {}, "fps": {}, "concurrency": {}}  # self-contained if the reference cell was skipped
# Measure TTFT and total latency with streaming
start_t = time.time()
first_token_t = None
n_content_chunks = 0
completion_tokens = None

stream = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "user",
         "content": [
            {"type": "video_url", "video_url": {"url": video_data_url}},
            {"type": "text", "text": "Describe this video in detail."},
         ]},
    ],
    max_tokens=512,
    temperature=0.4,
    top_p=0.4,
    stream=True,
    stream_options={"include_usage": True},   # final chunk carries token usage (honoured by NIM 1.7; the chunk count is the fallback)
)

for chunk in stream:
    if getattr(chunk, "usage", None):        # usage-only chunk at the end
        completion_tokens = chunk.usage.completion_tokens
        continue
    if not chunk.choices:
        continue
    if chunk.choices[0].delta.content:       # skip the role-only first delta
        if first_token_t is None:
            first_token_t = time.time()
        n_content_chunks += 1

total_time = time.time() - start_t
assert first_token_t is not None, "No tokens received - check the model server."
ttft = first_token_t - start_t
n_out = completion_tokens or n_content_chunks

print(f"Time to First Token (TTFT): {ttft:.2f}s")
print(f"Total Time: {total_time:.2f}s")
print(f"Generation Time: {total_time - ttft:.2f}s for {n_out} tokens (~{n_out / max(total_time - ttft, 1e-6):.1f} tok/s)")
bench.update(ttft_s=ttft, total_s=total_time, tok_per_s=n_out / max(total_time - ttft, 1e-6))


**What to expect:** TTFT is a large fraction of the total for a video request (the model must ingest thousands of vision tokens before emitting the first word), and the generation rate you measure is the number production sizing is built on - compare it with the reference run above if one is staged.

Now let's isolate the effect of **output length** by asking for the same video at three forced output lengths (1, 128 and 512 tokens). Input is identical each time; we force the model to keep generating until the budget is used, because a natural answer is often under 128 tokens and would hide the difference.


In [ ]:
bench = globals().get("bench") or {"osl": {}, "fps": {}, "concurrency": {}}  # self-contained if the reference cell was skipped
# OSL sweep - same input, different (forced) output lengths
osl_results = []
for osl in [1, 128, 512]:
    start_t = time.time()
    vlm_client.chat.completions.create(
        model=vlm_model,
        messages=[
            {"role": "user",
             "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": "Describe this video."},
             ]},
        ],
        max_tokens=osl,
        temperature=0.4,
        top_p=0.4,
        # Force the model to actually generate `osl` tokens; otherwise a short natural answer (~100 tokens) makes
        # 128 and 512 cost the same. vLLM options passed through the NIM (accepted by NIM 1.7).
        extra_body={"min_tokens": osl, "ignore_eos": True},
    )
    elapsed = time.time() - start_t
    osl_results.append((osl, elapsed))
    bench["osl"][str(osl)] = elapsed
    print(f"max_tokens={osl:4d}  ->  {elapsed:.2f}s")

print("\nOutput length drives total latency - the max_tokens=1 run is essentially pure TTFT, and a 1-token verdict costs a fraction of a full caption.")


**What to expect:** the `max_tokens=1` run takes roughly the TTFT you measured above; each additional output token adds a roughly constant increment, so 512 tokens costs several times the 1-token run.

---
#### 9.2 Input Size: Sampled Frames

The other side of the equation is input size. For video, the biggest input knob is **how many frames the model samples**. The NIM defaults to 4 FPS; we can override it per request with `media_io_kwargs`. We sweep *up to* the default and never above it, so the request stays under the ~16k vision-token guidance from Part 5 - watch `prompt_tokens` grow with FPS.


In [ ]:
bench = globals().get("bench") or {"osl": {}, "fps": {}, "concurrency": {}}  # self-contained if the reference cell was skipped
# Frame sampling sweep - fewer vs more frames from the same video (never above the 4 FPS default,
# which keeps the request under the ~16k multimodal-token guidance for this model).
for fps in [1, 2, 4]:
    start_t = time.time()
    response = vlm_client.chat.completions.create(
        model=vlm_model,
        messages=[
            {"role": "user",
             "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": "Describe this video in one paragraph."},
             ]},
        ],
        max_tokens=256,
        temperature=0.4,
        top_p=0.4,
        extra_body={"media_io_kwargs": {"video": {"fps": fps}}},
    )
    elapsed = time.time() - start_t
    prompt_tokens = response.usage.prompt_tokens if response.usage else "n/a"
    bench["fps"][str(fps)] = elapsed
    print(f"fps={fps}  ->  {elapsed:.2f}s  (prompt tokens: {prompt_tokens})")

print("\nMore sampled frames = more vision tokens = higher latency (but better temporal coverage).")


**What to expect:** prompt tokens scale roughly linearly with FPS (frames / 2 groups, each `H × W / 1024` tokens), and latency follows.

---
#### 9.3 Concurrency and Throughput

Finally, throughput: a serving engine batches concurrent requests, so total throughput grows with concurrency even as per-request latency degrades. Let's measure both by sending the same image request at increasing concurrency levels.


In [ ]:
bench = globals().get("bench") or {"osl": {}, "fps": {}, "concurrency": {}}  # self-contained if the earlier benchmark cells were skipped
from concurrent.futures import ThreadPoolExecutor

def one_request():
    start = time.time()
    vlm_client.chat.completions.create(
        model=vlm_model,
        messages=[
            {"role": "user",
             "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": "Describe this image in one paragraph."},
             ]},
        ],
        max_tokens=128,
        temperature=0.4,
        top_p=0.4,
    )
    return time.time() - start

print(f"{'Concurrency':>12} | {'Wall Time':>9} | {'Avg Latency':>11} | {'Throughput':>10}")
print("-" * 55)
for n in [1, 2, 4, 8, 16, 32]:
    start_t = time.time()
    with ThreadPoolExecutor(max_workers=n) as pool:
        latencies = list(pool.map(lambda _: one_request(), range(n)))
    wall = time.time() - start_t
    bench["concurrency"][str(n)] = {"wall_s": wall, "req_per_s": n / wall}
    print(f"{n:>12} | {wall:>8.2f}s | {sum(latencies)/len(latencies):>10.2f}s | {n/wall:>7.2f} req/s")


**What to expect:** at low concurrency the engine absorbs the extra requests almost for free; by 16 and 32 per-request latency climbs clearly, yet total throughput (requests per second) keeps climbing too - the serving engine is batching the work.

This is exactly how production sizing works. For example, the published [VSS RT-VLM performance tables](https://docs.nvidia.com/vss/3.2.0/performance-rt-vlm.html) show one H100 sustaining roughly **51 concurrent streams for single-token alerting (OSL = 1)** but only **33 for captioning (OSL = 100)**, measured at 10-second chunks, 80 frames at 448×448, with an FP8 8B reasoner checkpoint. Same GPU, same input - the difference is nothing but the OSL and concurrency trade-offs you just measured. You will see these stream counts again in the VSS alert-verification lab in Part 2.


---
### Part 10: Tear Down

*Estimated time: 1 minute - **required before Lab 2***

**Why:** while the NIM is running, its serving engine pre-allocates most of GPU 1's memory. Lab 2 loads Qwen Image Edit (~57 GB) onto **GPU 0** and the Cosmos 3 Nano generator onto **GPU 1** on the same two H100s, so a still-resident Reasoner NIM makes Lab 2 fail with out-of-memory errors. The cell below stops and removes the NIM container inside the lab's inner Docker daemon, then verifies that its GPU memory is actually free. The model cache lives in `/dli/task/data/cosmos3-reasoner-cache` and is preserved, so re-running Part 2 later brings the NIM back in a couple of minutes.

- **Run** the cell below. It is idempotent - safe to run twice or if the container is already gone.
- **Confirm** it prints `GPU memory is free - you can start Lab 2` before opening the next notebook.

On your own machine the equivalent is:

```bash
docker stop cosmos3-reasoner && docker rm cosmos3-reasoner
```

<div class="alert alert-block alert-danger"><b>Do not skip this cell.</b> Lab 2 will fail with out-of-memory errors if the NIM is still running.</div>


In [ ]:
# Tear down: stop + remove the NIM container (idempotent), then verify the GPU is really free.
state = container_state(NIM_CONTAINER)
if state is None:
    print(f"{NIM_CONTAINER} is not present - already torn down.")
else:
    print(f"Stopping {NIM_CONTAINER} (state: {state}) ...")
    r = docker_api("POST", f"/containers/{NIM_CONTAINER}/stop", params={"t": "30"}, timeout=120)
    print(f"  stop   -> HTTP {r.status_code}  (204 stopped, 304 already stopped)")
    r = docker_api("DELETE", f"/containers/{NIM_CONTAINER}", params={"force": "true", "v": "false"})
    print(f"  remove -> HTTP {r.status_code}  (204 removed, 404 already removed)")

assert container_state(NIM_CONTAINER) is None, f"{NIM_CONTAINER} is still present - re-run this cell or ask a TA."
print(f"Container {NIM_CONTAINER} removed. Cache '{NIM_CACHE}' preserved.")

# Verify 1: the endpoint must no longer answer.
try:
    requests.get(VLM_BASE_URL + "/v1/health/ready", timeout=3)
    raise RuntimeError(f"{VLM_BASE_URL} still answers - another container may be publishing this port. Ask a TA.")
except requests.RequestException:
    print(f"Endpoint {VLM_BASE_URL} is down.")

# Verify 2: GPU memory is released (the driver frees it within a few seconds of the process exiting).
if shutil.which("nvidia-smi"):
    for _ in range(12):
        out = subprocess.run(["nvidia-smi", f"--id={NIM_GPU}", "--query-gpu=memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True)
        if out.returncode != 0 or not out.stdout.strip():
            print("nvidia-smi query failed - but the endpoint is down, so the NIM process has exited.")
            print("GPU memory is free - you can start Lab 2.")
            break
        used_mib, total_mib = (int(x) for x in out.stdout.strip().split(","))
        if used_mib < 0.05 * total_mib:
            print(f"GPU {NIM_GPU}: {used_mib} / {total_mib} MiB in use.")
            print("GPU memory is free - you can start Lab 2.")
            break
        time.sleep(5)
    else:
        raise RuntimeError(f"GPU {NIM_GPU} still has {used_mib} MiB in use after 60 s. Ask a TA before starting Lab 2.")
else:
    # No nvidia-smi in this container: the failed health check above is our evidence that the NIM is gone.
    print("nvidia-smi not available here; the endpoint is down, which means the NIM process exited and released GPU memory.")
    print("GPU memory is free - you can start Lab 2.")


---
### Review

In this notebook you covered:
- What VLMs are and the **Cosmos 3** family (Edge / Nano / Super) - and how the Reasoner tower relates to the full model
- How the **Cosmos 3 Reasoner NIM** is deployed (one `docker run`, pinned to one GPU, weights from a pre-seeded cache), how to make sure it is running and healthy, and how to **stop it and verify the GPU is free**
- Image and video inference through the NIM's OpenAI-compatible API
- The vision token trade-off: more tokens means better quality, but slower inference and higher memory usage - and how `usage.prompt_tokens` proves it
- **Prompting**: role + focus + format + honesty beats a vague question every time
- **Structured outputs**: formatted reports, JSON events with timestamps (and checking that they parse), and single-token true/false and multiple-choice answers
- **Reasoning vs non-reasoning**: better grounding on complex judgments, at the cost of more generated tokens
- **Benchmarking** (optional): TTFT vs total latency, and how OSL, sampled frames, and concurrency drive performance


In the next notebook, we'll move from *understanding* video to *generating* it - using Cosmos for video generation and Qwen Image Edit to augment and build datasets, which we'll then use to fine-tune a VLM in the final notebook. Later, in the VSS lab in Part 2 of the course, the yes/no alerting pattern from Part 7 becomes VSS alert verification running on live camera streams.

Make sure Part 10 printed `GPU memory is free` before you open Lab 2.
